# 第 23 章　ロジスティック回帰

計測した米粒の形から，Cammeo と Osmancik をどこまで見分けられるでしょうか。まず長い方向の寸法を使い，Cammeo の確率を出します。次に形状を七列へ増やしたとき，確率と，品種を一つに決める判定がどう変わるかを確かめます。

前半は章本文に合わせて上から実行します。後半の約 20 分では検証の一粒を選び，予測の値と実際の品種を比べます。選んだ一粒だけでなく，近い確率を与えられた粒の集まりにも戻ります。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV が手元にない場合は公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

## 本文と一緒に実行

### 画像から表へ

図では，横の矢印が長軸，縦の矢印が短軸です。塗った部分を画素で数えると面積になります。これは説明用の模式図です。画像上で求めた長さや画素数を表にまとめたもので，元の写真や実物の mm での計測値はノートにはありません。

次の表で，予測に使える形状の列と，あとで正解と照合する品種の列を確認します。行番号は表の管理に使います。元の表は品種ごとに並んでいるため，品種を予測するときには行番号を使いません。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

fig, ax = plt.subplots(figsize=(7.2, 3.45), layout="constrained")
ax.add_patch(Ellipse((0, 0), 7, 2.6, facecolor="#e1ecf3", edgecolor="#195b8a", lw=1.8))
for x in [-3.5, 3.5]:
    ax.plot([x, x], [-1.85, 0], color="#7a858e", linestyle=":", lw=1)
for y in [-1.3, 1.3]:
    ax.plot([0, 4.05], [y, y], color="#7a858e", linestyle=":", lw=1)
span = {"arrowstyle": "<->", "color": "#195b8a", "lw": 1.5, "shrinkA": 0, "shrinkB": 0}
ax.annotate("", (3.5, -1.85), (-3.5, -1.85), arrowprops=span)
ax.annotate("", (4.05, 1.3), (4.05, -1.3), arrowprops=span)
ax.text(0, -2.28, "Major axis", ha="center", va="center")
ax.text(4.28, 0, "Minor axis", va="center")
ax.annotate("Area", (-1.6, 0.48), (-3.45, 2.0), ha="left",
            arrowprops={"arrowstyle": "->", "color": "#a7421f"})
ax.set(xlim=(-4.35, 5.95), ylim=(-2.7, 2.55), aspect="equal")
ax.axis("off")
plt.show()

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

file_name = "rice-grains.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

rice = pd.read_csv(csv_path)
train = rice.loc[rice["split"] == "train"]
valid = rice.loc[rice["split"] == "validation"]

columns_to_read = ["record_id", "major_axis", "minor_axis", "area", "variety"]
train[columns_to_read].head(3).round(2)

### 比較の基準と長さの分布

写真から形状の値を求め，その粒の品種を確認する前に予測します。この撮影で使われた粒から学習 2,286 行と検証 762 行を取り分けました。残る 762 行は保留にします。新しい収穫ロットや別の装置について評価するには，そうした条件での記録が要ります。

Cammeo を 1，Osmancik を 0 に固定します。いつも学習で多い方を答えると，検証では何粒で正解できますか。Cammeo は一粒でも見つかるでしょうか。

次の図は学習の粒について，各品種の中で長さの範囲に入る割合を描きます。どちらの山が右にあるかを見た後，二品種が重なるところを探してください。

In [ ]:
y_train = train["variety"].eq("Cammeo").astype(int)
y_valid = valid["variety"].eq("Cammeo").astype(int)

most_common = y_train.value_counts().idxmax()
baseline_probability = y_train.mean()
print(y_train.value_counts().sort_index())
print(f"学習の Cammeo の割合: {baseline_probability:.3f}")
print(f"一律に {most_common} と答えた検証の正解数: {(y_valid == most_common).sum()}")

In [ ]:
variety_styles = [
    ("Osmancik", "#195b8a", "--"),
    ("Cammeo", "#a7421f", "-"),
]
fig, ax = plt.subplots(figsize=(7.2, 3.65), layout="constrained")
for name, color, style in variety_styles:
    values = train.loc[train["variety"] == name, "major_axis"]
    ax.hist(values, bins=np.arange(140, 246, 5), weights=np.full(len(values), 100 / len(values)),
            histtype="step", linewidth=1.8, color=color, linestyle=style, label=name)
ax.set(xlabel="Major axis length (pixels)", ylabel="Share within variety (%)",
       xlim=(140, 245), xticks=[145, 165, 185, 205, 225, 245])
ax.legend(frameon=False, loc="upper right")
plt.show()

### スコアを確率へ変える

線形回帰のように入力からスコアを作り，S 字の関数で確率に変えます。`np.exp(x)` は約 2.718 の定数を x 乗する計算です。スコアが 0 のときは `np.exp(0)` が 1 なので，確率は `1 / (1 + 1) = 0.5` です。スコアを 2 ずつ動かしたときの確率の変化を，表で確かめてください。

確率から品種を一つ答えるときは，Cammeo の確率 0.5 以上を Cammeo とする規則を使います。0.51 と 0.99 の判定は同じでしょうか。個別に確認する順を決めるときにも，その二つを同じように扱うでしょうか。

In [ ]:
toy_score = pd.Series([-2, 0, 2, 4], name="score")
toy_probability = 1 / (1 + np.exp(-toy_score))
pd.DataFrame({"score": toy_score, "p_cammeo": toy_probability}).round(3)

### 長軸一本の予測確率

標準化とロジスティック回帰を一つの `Pipeline` にします。学習側の平均・標準偏差を求め，その目盛りに直した長軸の値をモデルに渡します。検証側も同じ目盛りを使います。`classes_` の順序を読み，`predict_proba()` の二列のどちらが Cammeo かを確認してください。

表では長軸が 170，190，210 の入力で，スコアと二品種の確率を並べます。図の上段の丸がその確率です。下段では，同じ長軸の位置に検証側の品種を並べています。確率が 0.5 になる長さの前後で，実際の二品種はどう並びますか。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_logistic_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(C=1.0, max_iter=2000)),
    ])


axis_features = ["major_axis"]
axis_model = make_logistic_model()
axis_model.fit(train[axis_features], y_train)
print(axis_model.classes_)

In [ ]:
probe = pd.DataFrame({"major_axis": [170, 190, 210]})
probe_probabilities = axis_model.predict_proba(probe)

probe.assign(
    score=axis_model.decision_function(probe),
    p_osmancik=probe_probabilities[:, 0],
    p_cammeo=probe_probabilities[:, 1],
).round(4)

In [ ]:
length_grid = pd.DataFrame({"major_axis": np.linspace(143, 242, 400)})
length_probability = axis_model.predict_proba(length_grid)[:, 1]
scale = axis_model.named_steps["scale"]
logistic = axis_model.named_steps["logistic"]
halfway = scale.mean_[0] - logistic.intercept_[0] * scale.scale_[0] / logistic.coef_[0, 0]

fig, (curve_ax, labels_ax) = plt.subplots(
    2, 1, figsize=(7.2, 4.8), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}, layout="constrained",
)
curve_ax.plot(length_grid["major_axis"], length_probability, color="#195b8a", lw=2)
curve_ax.scatter(probe["major_axis"], probe_probabilities[:, 1], color="#a7421f", zorder=3)
curve_ax.axhline(0.5, color="#7a858e", linestyle=":", linewidth=1)
curve_ax.set(ylabel="Predicted P(Cammeo)", ylim=(-0.06, 1.06), yticks=[0, 0.25, 0.5, 0.75, 1])
for ax in (curve_ax, labels_ax):
    ax.axvline(halfway, color="#7a858e", linestyle="--", linewidth=1)
for label, (name, color, _) in enumerate(variety_styles):
    values = valid.loc[valid["variety"] == name, "major_axis"]
    labels_ax.scatter(values, np.full(len(values), label), color=color, alpha=0.28,
                      s=13, marker="o" if label == 0 else "s", edgecolors="none")
labels_ax.set(xlabel="Major axis length (pixels)", yticks=[0, 1], ylim=(-0.45, 1.45),
              yticklabels=["Osmancik", "Cammeo"], xlim=(143, 242),
              xticks=[145, 160, 180, halfway, 200, 220, 240],
              xticklabels=["145", "160", "180", f"{halfway:.1f}", "200", "220", "240"])
plt.show()

### 形状を七つ使う

長軸一本で確率が近い二粒でも，短軸や面積が違えば，七列のモデルで確率が変わることがあります。同じ学習行を使い，入力を七つに増やしてもう一つのモデルを作ります。

検証の同じ 762 行に，長軸一本と七つの形状から出した Cammeo の確率を入れます。次に 0.5 以上なら 1 とする規則で，判定の列を作ります。確率と判定が別々の列であることを確認してください。

In [ ]:
features = [
    "area", "perimeter", "major_axis", "minor_axis",
    "eccentricity", "convex_area", "extent",
]
shape_model = make_logistic_model()
shape_model.fit(train[features], y_train)
print(shape_model.classes_)

In [ ]:
scored = valid.assign(
    actual=y_valid,
    p_axis=axis_model.predict_proba(valid[axis_features])[:, 1],
    p_shape=shape_model.predict_proba(valid[features])[:, 1],
)
threshold = 0.5
scored["pred_axis"] = (scored["p_axis"] >= threshold).astype(int)
scored["pred_shape"] = (scored["p_shape"] >= threshold).astype(int)

### 浅い木の確率

今度は同じ七入力を使い，元の目盛りのまま深さ 2 の木を作ります。図の `samples` は学習行数で，`value` は 0，1 の順の粒数です。長軸が 190 の粒が進む葉をたどり，学習行のうち Cammeo の割合を求めます。ここでは葉ごとに一定の確率が返ります。

最後の表では，同じ 0.5 の閾値を使って，どれだけの粒で正解したかを比べます。七つの形を渡したモデルにすると，長軸だけよりも正解数は必ず増えるでしょうか。

In [ ]:
from sklearn.tree import DecisionTreeClassifier

shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=2026)
shallow_tree.fit(train[features], y_train)
print(shallow_tree.classes_)

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(9, 4.65), layout="constrained")
plot_tree(shallow_tree, feature_names=features, impurity=False,
          rounded=True, precision=2, fontsize=11, ax=ax)
plt.show()

In [ ]:
scored["p_tree"] = shallow_tree.predict_proba(valid[features])[:, 1]
scored["pred_tree"] = (scored["p_tree"] >= threshold).astype(int)

correct = pd.Series({
    "Most common class": (scored["actual"] == most_common).sum(),
    "One feature": (scored["actual"] == scored["pred_axis"]).sum(),
    "Seven features": (scored["actual"] == scored["pred_shape"]).sum(),
    "Shallow tree": (scored["actual"] == scored["pred_tree"]).sum(),
}, name="correct")
pd.DataFrame({"correct": correct, "accuracy": correct / len(scored)}).round(3)

### 一粒と確率の帯

七列のモデルで Cammeo の確率が 0.9 に近い二粒を比べます。判定と実際の品種を照合し，確率が高いことから何が分かるか考えてください。

次に，検証の 762 粒を，七列のモデルの確率で三つの帯に分けます。それぞれの帯で，予測確率の平均と実際の Cammeo の割合を読みます。たとえば 0.8 以上の帯に，実際には Osmancik の粒が何粒入っていたでしょうか。

`pd.cut()` は，値が入る区間に名前を付けます。`right=False` は左端を含み，右端を外す指定です。0.2 ちょうどは真ん中の帯，0.8 ちょうどは最後の帯に入ります。最後の右端を `np.inf`（無限大）にして，1.0 も含めます。

In [ ]:
examples = [1086, 3799]
example_columns = ["major_axis", "minor_axis", "p_shape", "pred_shape", "variety"]
scored.set_index("record_id").loc[examples, example_columns].round(3)

In [ ]:
band_labels = ["0.2 未満", "0.2 以上 0.8 未満", "0.8 以上"]
probability_band = pd.cut(
    scored["p_shape"], bins=[0, 0.2, 0.8, np.inf], labels=band_labels, right=False,
)
scored["band"] = probability_band
band_summary = scored.groupby("band", observed=True).agg(
    rows=("actual", "size"),
    mean_probability=("p_shape", "mean"),
    cammeo=("actual", "sum"),
    observed_share=("actual", "mean"),
)
band_summary.round(3)

## 実習（約 20 分）

### 1. 一粒を選び，予想する（4 分）

次の六行から，気になる一粒を選びます。表では A〜F の目印と形状を表示します。学習側の長軸の図を手掛かりに，どちらの品種らしいか，それとも長軸は二品種の重なる範囲かを予想してください。

**選んだ目印と予想**　…

In [ ]:
practice_records = {"A": 3659, "B": 2905, "C": 413, "D": 3644, "E": 1754, "F": 858}
practice_shapes = scored.set_index("record_id").loc[
    list(practice_records.values()), ["major_axis", "minor_axis", "area"],
]
practice_shapes.index = pd.Index(list(practice_records), name="choice")
practice_shapes.round(2)

### 2. 確率と判定を照合する（9 分）

次のセルで `grain_choice` を選んだ目印に替えます。例えば A なら `"A"` と書きます。一列モデルと七列モデルそれぞれの Cammeo 確率，0.5 以上を 1 とした判定，実際の品種を照合します。確率がどちらへ動いたかと，予測した品種名も変わったかを別々に見てください。

続く表示は，選んだ粒と七列モデルの予測確率が近い検証行を集めた結果です。ここでは幅 0.2 の五つの帯を使い，講義よりも細かく見ます。例えば 0.2 以上 0.4 未満の帯には，ちょうど 0.2 の値も入ります。粒数と，実際に Cammeo だった粒数・割合を読み取ってください。

In [ ]:
grain_choice = None  # 選んだ目印に置き換える

if grain_choice is None:
    print("選んだ目印を grain_choice に指定してください。")
else:
    record_choice = practice_records[grain_choice]
    chosen = scored.loc[scored["record_id"] == record_choice]
    result_columns = [
        "record_id", "major_axis", "p_axis", "p_shape", "pred_axis", "pred_shape", "variety",
    ]
    display(chosen[result_columns].assign(threshold=threshold).round(4))

    practice_bands = pd.cut(
        scored["p_shape"], bins=[0, 0.2, 0.4, 0.6, 0.8, np.inf], right=False,
        labels=["0.2 未満", "0.2 以上 0.4 未満", "0.4 以上 0.6 未満",
                "0.6 以上 0.8 未満", "0.8 以上"],
    )
    chosen_band = practice_bands.loc[chosen.index].iloc[0]
    group = scored.loc[practice_bands == chosen_band]
    practice_report = pd.DataFrame([{
        "record_id": record_choice,
        "band": chosen_band,
        "rows": len(group),
        "mean_probability": group["p_shape"].mean(),
        "cammeo": group["actual"].sum(),
        "observed_share": group["actual"].mean(),
    }])
    display(practice_report.round(4))

### 3. 一粒と帯の結果を伝える（7 分）

最初の予想を振り返り，選んだ一粒の二つの確率と判定，実際の品種を使って説明します。次に，その粒と同じ確率帯の粒数・実際の Cammeo の割合を挙げ，一粒の正誤と帯全体の割合にはどんな違いがあるかを述べます。判定と一緒に確率を残す意味も含め，3〜4 文にまとめてください。

**一粒と確率帯の結果**　…

提出する場合は，報告に使った粒の目印をセルに指定し，カーネルを再起動して上から全セルを実行します。表と説明が残る状態でノートを保存してください。

### データの出典

Ilkay Cinar と Murat Koklu が公開した *Rice (Cammeo and Osmancik)* を使用します。教材用 CSV は列名を変更し，行番号と分割を追加しています。 [データ DOI](https://doi.org/10.24432/C5MW4Z) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)